# Distil Ksarth into Pathify's own small model (QLoRA on Kaggle)

Ksarth, Pathify's career guide, answers from cited career notes. This notebook teaches a small
open model to do the same job, so Pathify can run its guide without a paid API.

**Teacher → student.** `apps/ml/scripts/build_ksarth_dataset.py` ran Ksarth's real prompt and
retrieval, had Gemini answer, and kept only answers that passed automatic checks (valid citations,
no numbers absent from the notes, right language, sensible length). Here we fine-tune
**Qwen2.5-1.5B-Instruct** on those answers with **QLoRA**, measure it against the base model, and
export a **4-bit GGUF** that runs on a CPU with llama.cpp.

**Setup on Kaggle**
1. *Settings → Accelerator → GPU T4 ×2* (or P100), and *Internet → On*.
2. *Add-ons → Secrets*: add `HF_TOKEN`, a Hugging Face token with **write** access.
3. Put the dataset on the Hub from your machine:
   `huggingface-cli upload your-hf-username/pathify-ksarth-sft apps/ml/data/ksarth_sft --repo-type dataset --include "*.jsonl"`
   (or attach `train.jsonl` and `val.jsonl` to the notebook as a Kaggle dataset).
4. *Run all.* Training takes about 40-60 minutes on a T4.

In [ ]:
!pip -q install -U "transformers>=4.46" "trl>=0.12" "peft>=0.13" "bitsandbytes>=0.44" "accelerate>=1.1" datasets rouge-score

In [ ]:
import os, json, re, torch
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
HF_USER = "your-hf-username"                      # <- change
DATA_REPO = f"{HF_USER}/pathify-ksarth-sft"       # dataset repo with train.jsonl and val.jsonl
OUT_REPO = f"{HF_USER}/pathify-ksarth-1.5b"       # where the merged model and GGUF go
KAGGLE_DATA = "/kaggle/input/pathify-ksarth-sft"  # used instead if you attached the files to the notebook

In [ ]:
from datasets import load_dataset

if os.path.exists(KAGGLE_DATA):
    data = load_dataset("json", data_files={"train": f"{KAGGLE_DATA}/train.jsonl", "val": f"{KAGGLE_DATA}/val.jsonl"})
else:
    data = load_dataset(DATA_REPO, data_files={"train": "train.jsonl", "val": "val.jsonl"})
print(data)
print(data["train"][0]["messages"][1]["content"])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

tokenizer = AutoTokenizer.from_pretrained(BASE)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=quant, device_map="auto", torch_dtype=torch.float16)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

## Train

Loss is computed on the assistant's answer only (`assistant_only_loss`), so the model learns to
answer from the notes rather than to reproduce the long system prompt.

In [ ]:
from trl import SFTConfig, SFTTrainer

config = SFTConfig(
    output_dir="ksarth-lora",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_length=2048,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    assistant_only_loss=True,
    gradient_checkpointing=True,
    report_to=[],
)
trainer = SFTTrainer(model=model, args=config, train_dataset=data["train"], eval_dataset=data["val"],
                     processing_class=tokenizer, peft_config=lora)
trainer.train()
trainer.model.save_pretrained("ksarth-lora")

## Evaluate against the base model

On the held-out answers we check what matters for Ksarth, with the same rules that filtered the
teacher's data: citations that point at real notes, no numbers that aren't in the notes, the right
language, plus ROUGE-L against the teacher's answer.

In [ ]:
from rouge_score import rouge_scorer

NUMBER, CITE, DEVANAGARI = re.compile(r"\d+(?:\.\d+)?"), re.compile(r"\[(\d{1,2})\]"), re.compile(r"[\u0900-\u097f]")
SAFE = {"14416", "9", "10", "11", "12", "1", "2", "3"}
scorer = rouge_scorer.RougeScorer(["rougeL"])

def notes_in(system):
    return len(re.findall(r"^\[\d+\] ", system, flags=re.M))

def judge(system, question, answer, reference):
    n = notes_in(system)
    cites = [int(c) for c in CITE.findall(answer)]
    allowed = set(NUMBER.findall(system + " " + question)) | SAFE
    return {
        "valid_citations": all(1 <= c <= n for c in cites) and (bool(cites) or n == 0),
        "no_made_up_numbers": all(x in allowed for x in NUMBER.findall(answer)),
        "right_language": not DEVANAGARI.search(question) or bool(DEVANAGARI.search(answer)),
        "rougeL": scorer.score(reference, answer)["rougeL"].fmeasure,
    }

@torch.no_grad()
def answer(m, messages):
    prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to(m.device)
    out = m.generate(**ids, max_new_tokens=400, do_sample=False)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def evaluate(m, rows):
    scores = [judge(r["messages"][0]["content"], r["messages"][1]["content"], answer(m, r["messages"]), r["messages"][2]["content"]) for r in rows]
    return {k: round(sum(s[k] for s in scores) / len(scores), 3) for k in scores[0]}

val = list(data["val"])[:80]
tuned = evaluate(trainer.model, val)
with trainer.model.disable_adapter():
    base = evaluate(trainer.model, val)
report = {"base": base, "fine_tuned": tuned, "n": len(val)}
print(json.dumps(report, indent=1))
json.dump(report, open("distillation_report.json", "w"), indent=1)

## Merge, publish, and export a CPU-friendly GGUF

In [ ]:
from peft import AutoPeftModelForCausalLM

merged = AutoPeftModelForCausalLM.from_pretrained("ksarth-lora", torch_dtype=torch.float16, device_map="cpu").merge_and_unload()
merged.save_pretrained("ksarth-merged", safe_serialization=True)
tokenizer.save_pretrained("ksarth-merged")
merged.push_to_hub(OUT_REPO, token=os.environ["HF_TOKEN"])
tokenizer.push_to_hub(OUT_REPO, token=os.environ["HF_TOKEN"])

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip -q install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py ksarth-merged --outfile ksarth-f16.gguf --outtype f16
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF > /dev/null && cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!llama.cpp/build/bin/llama-quantize ksarth-f16.gguf pathify-ksarth-1.5b-Q4_K_M.gguf Q4_K_M

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
for f in ["pathify-ksarth-1.5b-Q4_K_M.gguf", "distillation_report.json"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=OUT_REPO)
print(f"https://huggingface.co/{OUT_REPO}")

## Serve it in Pathify

```bash
docker run -p 8080:8080 -v $PWD:/models ghcr.io/ggml-org/llama.cpp:server \
  -m /models/pathify-ksarth-1.5b-Q4_K_M.gguf --host 0.0.0.0 --port 8080 -c 4096
```

Then start the API with `LLM_PROVIDER=local` and `LOCAL_LLM_URL=http://localhost:8080/v1`.
Ksarth now answers with Pathify's own model; nothing else changes.